# Movies dataset

This notebook loads a small movie-domain graph into `NetworkXGraph`, `Neo4jGraph`,
`MemgraphGraph` and `JenaGraph`.

Dataset notes:
- movie nodes with title and IMDb id
- genre nodes linked with `IN_GENRE`
- source is a public sample API, with a bundled fallback for offline use

The server sections need a running server each: skip the ones you don't have. They read `NEO4J_*`, `MEMGRAPH_URL`/`MEMGRAPH_USER`/`MEMGRAPH_PASSWORD` and `FUSEKI_URL`/`FUSEKI_USER`/`FUSEKI_PASSWORD` (defaults: `bolt://localhost:7687` and `http://localhost:3030/ds`).

In [ ]:
import importlib.util

# Comprovació de presència del paquet, amb l'extra [notebook] del visualitzador de grafs (anywidget)
if (importlib.util.find_spec("cvcdocdb") is None
        or importlib.util.find_spec("cvcdocdb.viewer") is None
        or importlib.util.find_spec("anywidget") is None):
    print("⚠️ cvcdocdb[notebook] no està instal·lat. Iniciant instal·lació...")
    %pip install -q --upgrade "cvcdocdb[notebook]"
    print("✅ Instal·lació completada. L'estat del kernel PODRIA requerir un reinici.")
else:
    print("✅ cvcdocdb i anywidget ja són presents al sistema. Saltant instal·lació.")

In [ ]:
import os

from cvcdocdb import JenaGraph, MemgraphGraph, Neo4jGraph, NetworkXGraph
from cvcdocdb.exemples import load_movies_sample


def neo4j_config(default_target="DEV"):
    target = os.environ.get("NEO4J_TARGET", default_target).upper()
    prefix = f"NEO4J_{target}_"
    return {
        "target": target,
        "url": os.environ.get(f"{prefix}URL", os.environ.get("NEO4J_URL", "bolt://localhost:7687")),
        "user": os.environ.get(f"{prefix}USER", os.environ.get("NEO4J_USER", "neo4j")),
        "password": os.environ.get(f"{prefix}PASSWORD", os.environ.get("NEO4J_PASSWORD", "secret")),
        "database": os.environ.get(f"{prefix}DATABASE", os.environ.get("NEO4J_DATABASE", "neo4j")),
    }

## NetworkXGraph

In [ ]:
nx_graph = NetworkXGraph()
# load_movies_sample works with both NetworkXGraph (in-memory) and Neo4jGraph (persistent)
nx_stats = load_movies_sample(nx_graph, limit=25)
print("NetworkX stats:", nx_stats)

### Explore the graph

`show()` opens the cvcdocdb graph viewer on this graph. Click a node to see its properties,
double-click it to expand its neighbours, and search or filter from the sidebar. The viewer reads
the graph live from the kernel, so the graph is only closed in the last cell, once you are done.

In [ ]:
from cvcdocdb.viewer import show

show(nx_graph, load_all=True)

## Neo4jGraph

In [ ]:
cfg = neo4j_config()
print("Using Neo4j target:", cfg["target"])
neo4j_graph = Neo4jGraph(
    url=cfg["url"],
    user=cfg["user"],
    password=cfg["password"],
    database=cfg["database"],
)
try:
    neo4j_stats = load_movies_sample(neo4j_graph, limit=25)
    print("Neo4j stats:", neo4j_stats)
finally:
    neo4j_graph.close()


## MemgraphGraph

Same API and behaviour as `Neo4jGraph`.

In [ ]:
memgraph_url = os.environ.get("MEMGRAPH_URL", "bolt://localhost:7687")
print("Using Memgraph:", memgraph_url)
memgraph_graph = MemgraphGraph(
    memgraph_url,
    os.environ.get("MEMGRAPH_USER", ""),
    os.environ.get("MEMGRAPH_PASSWORD", ""),
)
try:
    memgraph_stats = load_movies_sample(memgraph_graph, limit=25)
    print("Memgraph stats:", memgraph_stats)
finally:
    memgraph_graph.close()

## JenaGraph

The graph is stored as RDF 1.2 in an Apache Jena Fuseki dataset (Fuseki >= 6.2.0).

In [ ]:
fuseki_url = os.environ.get("FUSEKI_URL", "http://localhost:3030/ds")
print("Using Fuseki:", fuseki_url)
jena_graph = JenaGraph(fuseki_url, os.environ.get("FUSEKI_USER"), os.environ.get("FUSEKI_PASSWORD"))
try:
    jena_stats = load_movies_sample(jena_graph, limit=25)
    print("Jena stats:", jena_stats)
finally:
    jena_graph.close()

## Cleanup

Closing the graph empties it, and the viewer with it: run this once you are done exploring.

In [ ]:
# nx_graph.close()